# We will run our eJTK analysis here

Analysis

In [1]:
from circadian_cell_painting import (
    detect,
    OscillationResults,
    eJTK_CYCLE,
    JTK_CYCLE
)
import polars as pl
from pathlib import Path
import numpy as np

# import multiprocessing as mp

# mp.set_start_method("fork", force=True)

# import logging

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s - %(levelname)s - %(message)s",
# )

# datasets

# --- WINDOWS -----------------------
# data_dir = Path(
#     "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data"
# )

# # platemaps
# pm_path = Path(
#     "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#     "Study/platemaps/2026-03-16CellPaint_map.xlsx"
# )
# platemap = pl.read_excel(pm_path, engine="openpyxl")
# # display(platemap)


# analysis_results_dir = Path(
#     "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/results"
# )

# pickle_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/pickles")

# visual_results_dir = Path(
#     "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals"
# )

# --- MAC OS -----------------------

data_dir = Path('/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Live '
                'Study/preprocessed_data')
pm_path = Path('/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Live '
               'Study/2026-03-16CellPaint_map.xlsx')
platemap = pl.read_excel(pm_path, engine="openpyxl")

analysis_results_dir = Path('/Users/dterciano/Desktop/PartchLabFiles/CircadianR'
                            'hythms/Live Study/results')
pickle_dir = Path('/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Liv'
                  'e Study/pickles')
visual_results_dir = Path('/Users/dterciano/Desktop/PartchLabFiles/CircadianRhy'
                          'thms/Live Study/visuals')


periods = np.arange(16, 48, dtype=int).tolist()

# model = eJTK_CYCLE(periods=periods, n_workers=20, n_perms=100)
model = JTK_CYCLE(periods=periods)

for dataset in data_dir.glob("*._.TIME-CORRECTED.parquet"):
    print(dataset)

    file_name = dataset.stem
    print(file_name)

    out_file = analysis_results_dir / f"{file_name}-JTK._.RESULTS.parquet"

    # I will be using median for this

    result: OscillationResults = detect(
        file_path=dataset,
        platemap_path=platemap,
        out_file=out_file,
        well_column="WellName",
        feature_col="feature",
        timepoint_col="Timepoint",
        value_col="median",
        stat="median",
        correction_group=["cell line", "condition"],
        detector=model,
    )

    pickle_out = pickle_dir / f"{file_name}-JTK._.RESULT_OBJ.pickle"
    result.to_pickle(pickle_out)

/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Live Study/preprocessed_data/2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED._.TIME-CORRECTED.parquet
2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED._.TIME-CORRECTED


Well progress: 100%|██████████| 12/12 [25:52<00:00, 129.35s/it]


/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Live Study/preprocessed_data/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED.parquet
2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED


Well progress: 100%|██████████| 12/12 [25:56<00:00, 129.74s/it]


In [ ]:
thresh = 0.05
drop_cols = ["jtk_fitted", "lsq_fitted"]

for file in analysis_results_dir.glob("*TIME-CORRECTED*.parquet"):
    out_csv = analysis_results_dir / f"{file.stem}.csv"
    print(out_csv)
    (pl.scan_parquet(file)
     .filter(pl.col("p_adjusted") <= thresh)
     .drop(drop_cols)
     .sink_csv(out_csv)
     )

/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Live Study/results/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED-JTK._.RESULTS.csv
/Users/dterciano/Desktop/PartchLabFiles/CircadianRhythms/Live Study/results/2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED._.TIME-CORRECTED-JTK._.RESULTS.csv


In [19]:
import polars as pl

test_res = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/results/2026-03-11CelPaint_PhenovNoTreatDex_HA"
                "RMONY_FIXED._.TIME-CORRECTED-JTK._.RESULTS.parquet")
# test_res = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#                 "Study/results/2026-03-10CelPaint_ChromaNoTreatDex._."
#                 "TIME-CORRECTED-JTK._.RESULTS.parquet")

lf = pl.scan_parquet(test_res).filter(pl.col("p_value") <=0.05)

display(lf.collect())

WellName,feature,stat,p_value,period,jtk_phase_hours,jtk_amplitude,tau,jtk_baseline,jtk_fitted,lsq_baseline,lsq_amplitude,lsq_phase_hours,lsq_r_squared,lsq_n_cycles,lsq_fitted,stain set,cell line,condition,p_adjusted
str,str,str,f64,f64,f64,f64,f64,f64,list[f64],f64,f64,f64,f64,f64,list[f64],str,str,str,f64
"""F8""","""NoBorderObj - Ring Region HOEC…","""median""",0.008582,25.0,0.0,0.012897,0.522727,0.001637,"[-0.003854, 0.005623, … -0.011158]",0.001481,0.007479,10.084442,0.061299,3.84,"[0.000236, -0.004475, … 0.00704]","""PhenoV""","""Bmal1""","""no treatment""",1.0
"""F9""","""NoBorderObj - Cell Region HOEC…","""median""",0.016754,17.0,14.5,0.001338,0.507576,0.000203,"[-0.000785, 0.00057, … 0.00008]",0.000277,0.00125,8.111753,0.049511,5.647059,"[0.001525, 0.000879, … -0.000517]","""PhenoV""","""Bmal1""","""no treatment""",1.0
"""F9""","""NoBorderObj - Ring Region Phen…","""median""",0.037178,23.0,22.0,0.008237,0.488636,0.00063,"[0.006253, 0.008868, … 0.008562]",0.001469,0.00069,17.149941,0.001409,4.173913,"[0.002073, 0.001637, … 0.00145]","""PhenoV""","""Bmal1""","""no treatment""",1.0
"""F9""","""NoBorderObj - Nucleus Region P…","""median""",0.00206,22.0,0.0,0.004001,0.55303,0.002603,"[0.005969, 0.006442, … 0.002033]",0.000702,0.003456,1.799991,0.042053,4.363636,"[0.002314, 0.004068, … 0.001955]","""PhenoV""","""Bmal1""","""no treatment""",1.0
"""F9""","""NoBorderObj - Nucleus Region H…","""median""",0.027166,18.0,15.5,0.00259,0.496212,0.000394,"[-0.002157, -0.001271, … 0.00128]",0.000424,0.000722,3.151321,0.011705,5.333333,"[0.000817, 0.000097, … -0.000297]","""PhenoV""","""Bmal1""","""no treatment""",1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""J8""","""NoBorderObj - Membrane Region …","""median""",0.027166,38.0,23.0,0.0,0.496212,-0.000004,"[-0.000004, -0.000004, … -0.000004]",-0.000003,0.000003,35.010566,0.04141,2.526316,"[-0.000002, -0.000004, … -0.000004]","""PhenoV""","""Per2""","""no treatment""",1.0
"""J8""","""NoBorderObj - Ring Region HOEC…","""median""",0.027166,38.0,23.0,0.0,0.496212,-0.000004,"[-0.000004, -0.000004, … -0.000004]",-0.000003,0.000003,35.010566,0.04141,2.526316,"[-0.000002, -0.000004, … -0.000004]","""PhenoV""","""Per2""","""no treatment""",1.0
"""J8""","""NoBorderObj - Nucleus Region H…","""median""",0.027166,38.0,23.0,0.0,0.496212,-0.000004,"[-0.000004, -0.000004, … -0.000004]",-0.000003,0.000003,35.010566,0.04141,2.526316,"[-0.000002, -0.000004, … -0.000004]","""PhenoV""","""Per2""","""no treatment""",1.0


Visualization (in-progress)